## QuickDraw Model Training and Setup

This notebook contains the model training and setup for the QuickDraw drawing recognition model. Here, the model is trained upon existing datasets that can be found in the "extracted_datasets" folder. For this project, the datasets are *apple, banana, carrot* and *circle, square, triangle*. So there are two model types (fruit and shapes) that you cna choose from. 

To create your own model trained on different shape categories, explore the links below to pull down your own data. Then, simply extract them as .npy files into the **extracted_datasets** folder, making sure to change filenames and paths if you start getting any 'File Not Found' errors.


1. Explore datasets here: (https://quickdraw.withgoogle.com/data)

2. Download (https://console.cloud.google.com/storage/browser/quickdraw_dataset/full/numpy_bitmap;tab=objects?inv=1&invt=AbxSNw&prefix=&forceOnObjectsSortingFiltering=false) a .npy models here.

3. View documentation here: (https://github.com/googlecreativelab/quickdraw-dataset)


QUICK NOTE: Went crazy after multiple Python 3.11 kernel crashes and decided to switch to version 3.10.0 to see if it's any better. Installed everything (numpy, tensorflow, sklearn) to the specific 3.10.0 version on 10/8/2025 to see if it helps.
UPDATE: looks like switching to version 3.10.0 solved the problem--no more kernel crashes, at least for the moment. Will NEED to use python 3.10.0 for all operations in this project from now on.
FURTHER UPDATE: This notebook now uses Python 3.12.2 as a "stable" primary kernel. If you experience issues, try going back to 3.10.0.

In [ ]:
!pip install pygame opencv-python # run this code cell if you are unsure whether or not you have pygame installed

  Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.2.6-cp311-cp311-win_amd64.whl.metadata (60 kB)
Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl (39.0 MB)
Using cached numpy-2.2.6-cp311-cp311-win_amd64.whl (12.9 MB)

  Attempting uninstall: numpy

    Found existing installation: numpy 1.26.4

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
    Uninstalling numpy-1.26.4:
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 

ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'C:\\Python311\\Scripts\\f2py.exe' -> 'C:\\Python311\\Scripts\\f2py.exe.deleteme'



**Troubleshooting Tip**: Depending on your install, you may need to uninstall and reinstall the correct version of numpy for this project. You can safely ignore this series of cells if you aren't experiencing any issues.

In [3]:
!pip uninstall numpy

^C


In [ ]:
!pip install numpy==1.26.4

In [ ]:
!pip install tensorflow==2.14.0

  Using cached tensorflow-2.14.0-cp311-cp311-win_amd64.whl.metadata (3.3 kB)
  Using cached tensorflow_intel-2.14.0-cp311-cp311-win_amd64.whl.metadata (4.8 kB)
Using cached tensorflow-2.14.0-cp311-cp311-win_amd64.whl (2.1 kB)
Using cached tensorflow_intel-2.14.0-cp311-cp311-win_amd64.whl (284.2 MB)

   ---------------------------------------- 0/2 [tensorflow-intel]
   ---------------------------------------- 0/2 [tensorflow-intel]
   ---------------------------------------- 0/2 [tensorflow-intel]
   ---------------------------------------- 0/2 [tensorflow-intel]
   ---------------------------------------- 0/2 [tensorflow-intel]
   ---------------------------------------- 0/2 [tensorflow-intel]
   ---------------------------------------- 0/2 [tensorflow-intel]
   ---------------------------------------- 0/2 [tensorflow-intel]
   ---------------------------------------- 0/2 [tensorflow-intel]
   ---------------------------------------- 0/2 [tensorflow-intel]
   --------------------------

ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'C:\\Python311\\Scripts\\estimator_ckpt_converter.exe' -> 'C:\\Python311\\Scripts\\estimator_ckpt_converter.exe.deleteme'



## Installing pygame

Pygame is a series of modules and libraries designed to creat games and apps. For this lesson you will be integrating your AI project with Pygame to create a canvas where you can draw.

1. Run the cell below to install pygame to your project.

In [ ]:
!pip install pygame opencv-python

  Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.2.6-cp311-cp311-win_amd64.whl.metadata (60 kB)
Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl (39.0 MB)
Using cached numpy-2.2.6-cp311-cp311-win_amd64.whl (12.9 MB)

  Attempting uninstall: numpy

    Found existing installation: numpy 1.26.4

    Uninstalling numpy-1.26.4:

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
      Successfully uninstalled numpy-1.26.4
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   

ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'C:\\Python311\\Scripts\\f2py.exe' -> 'C:\\Python311\\Scripts\\f2py.exe.deleteme'



## Training AI Model.

Next you will create the script to run and your AI model. Please view iD Game Plan on how to build this script.

1.  First install the dendencies and libraries for this project.

In [1]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
import os

2. Create Constant Variables for your epochs and class names.
      Replace your CLASS_NAMES with the names of your datasets your downloaded.

In [2]:
EPOCHS = 35
CLASS_NAMES= ['circle', 'square', 'triangle']

3. Load your datasets and label them.

In [3]:
images = []
labels = []

for index, name in enumerate(CLASS_NAMES):
    data = np.load(f"extracted_datasets/full_numpy_bitmap_{name}.npy")[:3000] / 255.0
    label = np.full(len(data), index)
    images.append(data)
    labels.append(label)

4. Combine then split your data

In [9]:
X = np.concatenate(images).reshape(-1, 28, 28, 1)
y = np.concatenate(labels)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

5. Build your model

In [7]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, (3,3), activation='relu', input_shape=(28,28,1)),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax')
])

C:\Users\car-m\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


6. Compile and train your model.

In [10]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=EPOCHS, validation_data=(X_test, y_test))

Epoch 1/35
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9281 - loss: 0.2122 - val_accuracy: 0.9744 - val_loss: 0.0790
Epoch 2/35
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9699 - loss: 0.0897 - val_accuracy: 0.9806 - val_loss: 0.0596
Epoch 3/35
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9751 - loss: 0.0734 - val_accuracy: 0.9728 - val_loss: 0.0835
Epoch 4/35
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9774 - loss: 0.0660 - val_accuracy: 0.9789 - val_loss: 0.0576
Epoch 5/35
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9794 - loss: 0.0573 - val_accuracy: 0.9806 - val_loss: 0.0551
Epoch 6/35
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9826 - loss: 0.0490 - val_accuracy: 0.9783 - val_loss: 0.0631
Epoch 7/35
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9843 - loss: 0.0438 - val_accuracy: 0.9728 - val_loss: 0.0861
Epoch 8/35
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9869 - loss: 0.0385 - val_accuracy: 0.

6. Save your model in your models folder.

In [ ]:
os.makedirs("models", exist_ok=True)
model.save("models/quickdraw_model_shapes_v2.keras") # change the name of your model if you changed the categories it's trained on (fruit, shapes, etc.)

7. Create the program to run drawpad.py, or just go directly to the **drawpad.py** file in this same directory if you're experiencing issues with the code cell below. Once the pop-up window launches, you'll be able to start drawing your objects.

In [6]:
# QUICK HISTORICAL INFO: after several issues with this program and package compatibility issues,
# several packages were updated/uninstalled so that the pip-check command would not produce
# package mismatch issues. The program is now working.

import os
import subprocess

script_path = os.path.abspath("Unit3/quickdraw_model/drawpad.py")
subprocess.Popen(f'start cmd /c python "{script_path}"', shell=True)

<Popen: returncode: None args: 'start cmd /c python "c:\\Users\\car-m\\machi...>